---
title: "单调栈"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

## 📝 题目 739：每日温度

::: {.callout-caution}
### 📖 题目描述
给定一个整数数组 `temperatures` ，表示每天的温度，返回一个数组 `answer` ，其中 `answer[i]` 是指对于第 `i` 天，下一个更高温度出现在几天后。如果气温在这之后都不会升高，请在该位置用 `0` 来代替。

**输入输出示例**：

* **示例 1**：
    * **输入**：`temperatures = [73,74,75,71,69,72,76,73]`
    * **输出**：`[1,1,4,2,1,1,0,0]`

* **示例 2**：
    * **输入**：`temperatures = [30,40,50,60]`
    * **输出**：`[1,1,1,0]`

* **示例 3**：
    * **输入**：`temperatures = [30,60,90]`
    * **输出**：`[1,1,0]`
:::

In [8]:
class Solution739:
    def dailyTemperatures(self, temperatures: list[int]) -> list[int]:
        n = len(temperatures)
        # 1. 初始化结果数组为全 0
        ans = [0] * n
        # 2. 单调栈，存放的是索引
        stack = []

        for i, t in enumerate(temperatures):
            # 3. 当发现当前温度比栈顶温度高时
            while stack and temperatures[stack[-1]] < t:
                prev_index = stack.pop()  # 弹出栈顶，说明它等到了更高温
                ans[prev_index] = i - prev_index  # 计算天数差并存入答案
            # 4. 当前索引入栈（作为新的等待者）
            stack.append(i)

        return ans

In [10]:
#| code-fold: true
def test_739(func):
    test_cases = [
        {"desc": "1. 示例 1: 经典波动", "temps": [73,74,75,71,69,72,76,73], "expected": [1,1,4,2,1,1,0,0]},
        {"desc": "2. 示例 2: 严格递增", "temps": [30,40,50,60], "expected": [1,1,1,0]},
        {"desc": "3. 示例 3: 跨度极大", "temps": [30,60,90], "expected": [1,1,0]},
        {"desc": "4. 边界: 只有一个元素", "temps": [30], "expected": [0]},
        {"desc": "5. 严格递减 (全员等不到)", "temps": [90,80,70,60], "expected": [0,0,0,0]},
        {"desc": "6. 全部温度相同", "temps": [70,70,70,70], "expected": [0,0,0,0]},
        {"desc": "7. U型回升", "temps": [80,70,60,70,80], "expected": [0,3,1,1,0]},
        {"desc": "8. 锯齿状频繁波动", "temps": [30,31,30,31,30,31], "expected": [1,0,1,0,1,0]},
        {"desc": "9. 包含极端高温", "temps": [30,30,30,100], "expected": [3,2,1,0]},
        {"desc": "10. 大型平滑波段", "temps": [50,51,52,53,50,50,50,54], "expected": [1,1,1,4,3,2,1,0]}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<20} | {'实际':<20}")
    print("-" * 80)

    for i, tc in enumerate(test_cases):
        actual = func(tc["temps"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1

        e_str = str(tc['expected'])[:20] + "..." if len(str(tc['expected'])) > 20 else str(tc['expected'])
        a_str = str(actual)[:20] + "..." if len(str(actual)) > 20 else str(actual)

        print(f"{i+1:<3} | {status:<6} | {e_str:<20} | {a_str:<20} | {tc['desc']}")

    print("-" * 80)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_739(Solution739().dailyTemperatures)

ID  | 结果     | 预期                   | 实际                  
--------------------------------------------------------------------------------
1   | ✅ PASS | [1, 1, 4, 2, 1, 1, 0... | [1, 1, 4, 2, 1, 1, 0... | 1. 示例 1: 经典波动
2   | ✅ PASS | [1, 1, 1, 0]         | [1, 1, 1, 0]         | 2. 示例 2: 严格递增
3   | ✅ PASS | [1, 1, 0]            | [1, 1, 0]            | 3. 示例 3: 跨度极大
4   | ✅ PASS | [0]                  | [0]                  | 4. 边界: 只有一个元素
5   | ✅ PASS | [0, 0, 0, 0]         | [0, 0, 0, 0]         | 5. 严格递减 (全员等不到)
6   | ✅ PASS | [0, 0, 0, 0]         | [0, 0, 0, 0]         | 6. 全部温度相同
7   | ✅ PASS | [0, 3, 1, 1, 0]      | [0, 3, 1, 1, 0]      | 7. U型回升
8   | ✅ PASS | [1, 0, 1, 0, 1, 0]   | [1, 0, 1, 0, 1, 0]   | 8. 锯齿状频繁波动
9   | ✅ PASS | [3, 2, 1, 0]         | [3, 2, 1, 0]         | 9. 包含极端高温
10  | ✅ PASS | [1, 1, 1, 4, 3, 2, 1... | [1, 1, 1, 4, 3, 2, 1... | 10. 大型平滑波段
--------------------------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题本质上是：**为数组中的每个元素，寻找右边第一个比它大的元素。**

**单调栈的降维打击：**
我们维护一个**单调递减栈**，栈里存放的是日期的**索引**。

1. **入栈逻辑（入队等待）**：
   遍历每一天的气温。如果当前气温比栈顶气温矮（或相等），说明这一天还没等到更高的气温。把它作为“等待者”压入栈中。
2. **出栈逻辑（精准报喜）**：
   如果当前气温 `T[i]` 比栈顶气温 `T[stack[-1]]` 高，说明栈顶那一天的“出头之日”到了！
   - 弹出栈顶索引 `prev_index`。
   - 计算天数差距：`i - prev_index`。
   - 将这个差距填入结果数组 `answer[prev_index]`。
   - 继续检查新的栈顶，直到栈空或当前气温不再比栈顶高。
3. **默认值处理**：
   结果数组初始化为全 `0`。如果某一天直到最后都没被弹出栈（说明后面没气温比它高），它在结果数组里的值就自然保持为 `0`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：O(N)。虽然有嵌套的 while 循环，但每根“温度计”只会被压入栈一次、弹出栈一次。所有操作相加是线性的。
* **空间复杂度**：O(N)。栈在最坏情况下（温度一路走低）会存放所有天数的索引。
:::

## 📝 题目 84：柱状图中最大的矩形

::: {.callout-caution}
### 📖 题目描述
给定 `n` 个非负整数，用来表示柱状图中各个柱子的高度。每个柱子彼此相邻，且宽度为 1。
求在该柱状图中，能够勾勒出来的矩形的最大面积。

**输入输出示例**：

* **示例 1**：
    * **输入**：`heights = [2,1,5,6,2,3]`
    * **输出**：`10`
    * **解释**：最大的矩形是高度为 5 和 6 的那两根柱子组合而成的，面积为 5 * 2 = 10。

* **示例 2**：
    * **输入**：`heights = [2,4]`
    * **输出**：`4`
:::

In [1]:
class Solution84:
    def largestRectangleArea(self, heights: list[int]) -> int:
        heights = [0] + heights + [0]  # 头部 0：防止栈空，提供永远的“左边界”，尾部 0：强迫所有遗留在栈里的柱子在最后全部出栈结算
        stack = [] # 存放柱子的索引，保持高度单调递增
        max_area = 0

        for i, h in enumerate(heights):
            while stack and heights[stack[-1]] > h:  # 当遇到比栈顶矮的柱子，说明栈顶柱子的右边界找到了，开始清算
                curr_height = heights[stack.pop()]   # 1. 获取要清算的柱子高度
                curr_width = i - stack[-1] - 1  # 2. 此时的新栈顶，就是它的左边界，当前的 i 就是它的右边界
                max_area = max(max_area, curr_height * curr_width)   # 3. 结算面积
            stack.append(i)  # 清算完毕，当前柱子入栈

        return max_area

In [4]:
#| code-fold: true
def test_84(func):
    test_cases = [
        {"desc": "1. 示例 1: 经典锯齿形", "heights": [2,1,5,6,2,3], "expected": 10},
        {"desc": "2. 示例 2: 简单递增", "heights": [2,4], "expected": 4},
        {"desc": "3. 边界: 空数组 (题目限制至少为1,这里做鲁棒性测试)", "heights": [], "expected": 0},
        {"desc": "4. 边界: 单个柱子", "heights": [5], "expected": 5},
        {"desc": "5. 全部高度相同", "heights": [2,2,2,2], "expected": 8},
        {"desc": "6. 严格递增序列 (考验尾部哨兵)", "heights": [1,2,3,4,5], "expected": 9},
        {"desc": "7. 严格递减序列", "heights": [5,4,3,2,1], "expected": 9},
        {"desc": "8. V字型陷阱", "heights": [5,4,1,4,5], "expected": 8},
        {"desc": "9. 包含 0 的断层测试", "heights": [2,1,2,0,3,2,2,3], "expected": 8}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<6} | {'实际':<6} | {'描述'}")
    print("-" * 75)

    # 针对空数组做个小防护，虽然题干说 n>=1
    for i, tc in enumerate(test_cases):
        if not tc["heights"]:
            actual = 0
        else:
            actual = func(tc["heights"].copy()) # 传入 copy 防止原数组被修改导致断言干扰

        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<6} | {str(actual):<6} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_84(Solution84().largestRectangleArea)

ID  | 结果     | 预期     | 实际     | 描述
---------------------------------------------------------------------------
1   | ✅ PASS | 10     | 10     | 1. 示例 1: 经典锯齿形
2   | ✅ PASS | 4      | 4      | 2. 示例 2: 简单递增
3   | ✅ PASS | 0      | 0      | 3. 边界: 空数组 (题目限制至少为1,这里做鲁棒性测试)
4   | ✅ PASS | 5      | 5      | 4. 边界: 单个柱子
5   | ✅ PASS | 8      | 8      | 5. 全部高度相同
6   | ✅ PASS | 9      | 9      | 6. 严格递增序列 (考验尾部哨兵)
7   | ✅ PASS | 9      | 9      | 7. 严格递减序列
8   | ✅ PASS | 8      | 8      | 8. V字型陷阱
9   | ✅ PASS | 8      | 8      | 9. 包含 0 的断层测试
---------------------------------------------------------------------------
测试总结: 通过 9/9


::: {.callout-important}
### 💡 思路讲解

这道题的暴力解法是：遍历每一根柱子，分别向左和向右找，直到找到第一个比它矮的柱子，这就是它的左右边界。时间复杂度 O(N^2)，直接超时。

**单调栈的 O(N) 降维打击：**
我们需要一个**单调递增栈**（栈里存放索引，对应的高度必须越来越高）。

1. **入栈规则（安全期）**：
   只要新来的柱子比栈顶柱子高，说明栈顶柱子的“右边界”还没找到，它还能继续向右扩张！把它乖乖压入栈中。
2. **出栈规则（边界确认，终极清算）**：
   当新来的柱子 `i` 比栈顶柱子矮时，**神迹降临**！栈顶柱子的“右边界”被找到了，就是当前的 `i`！
   把它弹出栈。此时，弹完之后的**新栈顶**，就是它左边第一个比它矮的柱子，也就是它的“左边界”！
   有了左右边界，高度又是它自己，面积瞬间算出：`高度 * (右边界 - 左边界 - 1)`。
3. **黑魔法：首尾哨兵技巧 (Sentinel)**：
   - 如果数组是单调递增的（如 `[1,2,3]`），所有的柱子都会留在栈里无法结算。
   - 如果遇到第一根柱子，它弹出来后栈空了，左边界怎么算？
   - **绝杀技巧**：在原数组的**最前面和最后面各加一个高度为 0 的哨兵**！
   - 前面的 0 保证了栈永远不为空（完美的左边界地基）；后面的 0 保证了所有柱子在最后一定会被强制弹出并结算！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：O(N)。虽然代码里有一个 while 循环，但每根柱子的索引最多入栈一次、出栈一次。没有任何重复劳动，均摊下来是绝对的线性时间。
* **空间复杂度**：O(N)。最坏情况下（如柱子高度单调递增），栈内会存放所有的索引。
:::

## 📝 题目 42：接雨水

::: {.callout-caution}
### 📖 题目描述
给定 `n` 个非负整数表示每个宽度为 1 的柱子的高度图，计算按此排列的柱子，下雨之后能接多少雨水。

**输入输出示例**：

* **示例 1**：
    * **输入**：`height = [0,1,0,2,1,0,1,3,2,1,2,1]`
    * **输出**：`6`
    * **解释**：上面是由数组 [0,1,0,2,1,0,1,3,2,1,2,1] 表示的高度图，在这种情况下，可以接 6 单位的雨水（蓝色部分表示雨水）。

* **示例 2**：
    * **输入**：`height = [4,2,0,3,2,5]`
    * **输出**：`9`
:::

In [5]:
class Solution42:
    def trap(self, height: list[int]) -> int:
        if not height:
            return 0

        stack = []  # 存放柱子的索引，保持高度单调递减
        total_water = 0

        for i, h in enumerate(height):
            # 当遇到比栈顶高的柱子，说明凹槽的右边界找到了，开始清算存水！
            while stack and height[stack[-1]] < h:
                # 1. 弹出底部的柱子索引
                top = stack.pop()

                # 如果弹完栈空了，说明没有左边界，无法形成凹槽，跳过
                if not stack:
                    break

                # 2. 获取新的栈顶作为左边界
                left = stack[-1]

                # 3. 计算本层积水的体积 (水池高度 * 水池宽度)
                # 高度 = min(左边界高度, 右边界高度) - 底部高度
                bounded_height = min(height[left], h) - height[top]
                # 宽度 = 右边界索引 - 左边界索引 - 1
                bounded_width = i - left - 1

                # 累加本层积水
                total_water += bounded_height * bounded_width

            # 清算完毕（或保持了单调递减），当前柱子入栈
            stack.append(i)

        return total_water

In [6]:
#| code-fold: true
def test_42(func):
    test_cases = [
        {"desc": "1. 示例 1: 经典过山车高度图", "height": [0,1,0,2,1,0,1,3,2,1,2,1], "expected": 6},
        {"desc": "2. 示例 2: M型高度图", "height": [4,2,0,3,2,5], "expected": 9},
        {"desc": "3. 边界: 空数组", "height": [], "expected": 0},
        {"desc": "4. 边界: 单个柱子", "height": [5], "expected": 0},
        {"desc": "5. 边界: 两个柱子", "height": [5,3], "expected": 0},
        {"desc": "6. 全部高度相同 (平滑地面)", "height": [2,2,2,2], "expected": 0},
        {"desc": "7. 严格递增序列 (斜坡)", "height": [1,2,3,4,5], "expected": 0},
        {"desc": "8. 严格递减序列 (滑梯)", "height": [5,4,3,2,1], "expected": 0},
        {"desc": "9. V字型陷阱 (中间是谷底)", "height": [5,0,5], "expected": 5},
        {"desc": "10. 带 0 的断层存水测试", "height": [0,1,0,2,1,0,1,3,2,1,2,1], "expected": 6}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        # 传入 copy 防止原数组被修改
        actual = func(tc["height"].copy())

        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<4} | {str(actual):<4} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_42(Solution42().trap)

ID  | 结果     | 预期   | 实际   | 描述
---------------------------------------------------------------------------
1   | ✅ PASS | 6    | 6    | 1. 示例 1: 经典过山车高度图
2   | ✅ PASS | 9    | 9    | 2. 示例 2: M型高度图
3   | ✅ PASS | 0    | 0    | 3. 边界: 空数组
4   | ✅ PASS | 0    | 0    | 4. 边界: 单个柱子
5   | ✅ PASS | 0    | 0    | 5. 边界: 两个柱子
6   | ✅ PASS | 0    | 0    | 6. 全部高度相同 (平滑地面)
7   | ✅ PASS | 0    | 0    | 7. 严格递增序列 (斜坡)
8   | ✅ PASS | 0    | 0    | 8. 严格递减序列 (滑梯)
9   | ✅ PASS | 5    | 5    | 9. V字型陷阱 (中间是谷底)
10  | ✅ PASS | 6    | 6    | 10. 带 0 的断层存水测试
---------------------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题的暴力解法是：对于每一根柱子，分别向左和向右找，直到找到左边最高的和右边最高的柱子。它能接的水量 = `min(left_max, right_max) - 自己高度`。时间复杂度 O(N^2)，直接超时。

**单调栈的 O(N) 降维打击：**
我们需要一个**单调递减栈**（栈里存放索引，对应的高度必须越来越矮）。这就像是在模拟一个从左向右扫描的“凹槽检测器”。

1. **入栈规则（安全期）**：
   只要新来的柱子比栈顶柱子矮，说明它们之间可以形成一个潜在的“凹槽”来存水！把它的索引乖乖压入栈中。
2. **出栈规则（边界确认，积水结算）**：
   当新来的柱子 `i` 比栈顶柱子高时，**神迹降临**！栈顶柱子 `top` 找到了它的“右边界”（就是当前的 `i`）！
   把它弹出栈。此时，弹完之后的**新栈顶** `left`，就是它左边第一个比它高的柱子，也就是它的“左边界”！
   有了左右边界和它自己的高度（`height[top]`），我们就可以计算出这一层积水的体积：
   - **水池高度** = `min(height[left], height[i]) - height[top]` （注意：必须是个“凹槽”，所以要减去底部的柱子高度）。
   - **水池宽度** = `i - left - 1`。
   - **本层体积** = 水池高度 * 水池宽度。
   将体积累加到结果中。
3. **注意**：如果弹完栈顶后栈空了，说明没有左边界，无法形成凹槽，不计存水。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：O(N)。虽然代码里有一个 while 循环，但每根柱子的索引最多入栈一次、出栈一次。没有任何重复劳动，均摊下来是绝对的线性时间。
* **空间复杂度**：O(N)。最坏情况下（如柱子高度单调递减），栈内会存放所有的索引。
:::